In [1]:
# make_splits.py

# Builds leakage-safe splits from signatures_labeled.csv (output of
# label_signatures.py).

# Grouping rule: every signature with the same (normalized) name stays in the
# same partition. Names that are near-duplicates of each other (for example
# "gayathri" / "pgayathri") are merged into one group as well.

# NOTE: do NOT group on the `signer_id` column written by label_signatures.py.
# Only file names like 1-003_08.png carry a real signer ID. For names like
# 54_0651.png the leading number is a sample index, not a person.

# Outputs splits.csv with one row per signature:
#     file name, name, group, source_family, label,
#     split            ('test' or 'dev')
#     cv_seed42/43/44  (fold 0-4 for dev rows, -1 for test rows)

# Requires: pandas, numpy, scikit-learn, networkx, rapidfuzz

In [2]:

import re

import networkx as nx
import numpy as np
import pandas as pd
from rapidfuzz.distance import Levenshtein
from sklearn.model_selection import StratifiedGroupKFold

# ----------------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------------
IN_CSV = "signatures_labeled.csv"
OUT_CSV = "splits.csv"
SIM_COL = "sim_basic"            # or "sim_digit_map"
NAME_COL = "expected_basic"      # normalized expected name
LABEL_CUTOFF = 80                # legible if similarity >= this
NEAR_DUP = 0.75                  # merge names at least this similar (0-1)
N_SPLITS = 5
TEST_SEED = 42                   # fixes the held-out test groups
CV_SEEDS = (42, 43, 44)          # repeated grouped CV on the dev portion


def source_family(fname):
    """A = signer-sample ids (1-003_08), B = N_NNNN, C = bare number."""
    if re.match(r"^\d+-\d+_\d+\.\w+$", fname):
        return "A"
    if re.match(r"^\d+_\d+\.\w+$", fname):
        return "B"
    return "C"


def build_groups(names):
    """Connected components over names; edges join near-duplicate names."""
    uniq = sorted(set(names))
    g = nx.Graph()
    g.add_nodes_from(uniq)
    merged = []
    for i, a in enumerate(uniq):
        for b in uniq[i + 1:]:
            sim = 1 - Levenshtein.distance(a, b) / max(len(a), len(b))
            if sim >= NEAR_DUP:
                g.add_edge(a, b)
                merged.append((a, b, round(sim, 2)))
    comp = {n: i for i, c in enumerate(nx.connected_components(g)) for n in c}
    return comp, merged


def describe(df, title):
    print(f"\n{title}")
    print(f"  rows {len(df)}, legible {int(df.label.sum())} ({df.label.mean():.1%}), "
          f"groups {df.group.nunique()}")


def main():
    df = pd.read_csv(IN_CSV, keep_default_na=False)
    df["name"] = df[NAME_COL].astype(str)
    df["label"] = (df[SIM_COL].astype(float) >= LABEL_CUTOFF).astype(int)
    df["source_family"] = df["file name"].map(source_family)

    comp, merged = build_groups(df["name"])
    df["group"] = df["name"].map(comp)

    print(f"{len(df)} signatures, {df.name.nunique()} distinct names, "
          f"{df.group.nunique()} groups after merging near-duplicate names")
    print("Merged name pairs (check these by eye):")
    for a, b, s in merged:
        print(f"   {a} ~ {b}  ({s})")
    print("Group size (rows): min %d, median %d, max %d" % (
        df.groupby("group").size().min(),
        df.groupby("group").size().median(),
        df.groupby("group").size().max()))

    # ---- Label mix per name and per source family -------------------------
    per_name = df.groupby("name")["label"].mean()
    print(f"\nNames all illegible: {(per_name == 0).sum()}, "
          f"all legible: {(per_name == 1).sum()}, "
          f"mixed: {((per_name > 0) & (per_name < 1)).sum()}")
    print("\nLegible rate by source family (possible confound):")
    print(df.groupby("source_family")["label"].agg(rows="size", legible_rate="mean")
          .round(3).to_string())

    # ---- Held-out in-distribution test: one fold of an outer split --------
    outer = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=TEST_SEED)
    dev_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
    df["split"] = "dev"
    df.loc[df.index[test_idx], "split"] = "test"
    assert set(df.loc[df.split == "dev", "group"]).isdisjoint(
        df.loc[df.split == "test", "group"])
    describe(df[df.split == "dev"], "DEV (used for model selection)")
    describe(df[df.split == "test"], "TEST (touch once, at the very end)")

    # ---- Repeated grouped CV on dev only -----------------------------------
    dev = df[df.split == "dev"]
    for seed in CV_SEEDS:
        col = f"cv_seed{seed}"
        df[col] = -1
        cv = StratifiedGroupKFold(N_SPLITS, shuffle=True, random_state=seed)
        print(f"\nCV seed {seed}: fold sizes / legible / groups")
        for k, (tr, va) in enumerate(cv.split(dev, dev["label"], dev["group"])):
            assert set(dev.group.iloc[tr]).isdisjoint(dev.group.iloc[va])
            df.loc[dev.index[va], col] = k
            v = dev.iloc[va]
            print(f"   fold {k}: {len(v):4d} rows, {int(v.label.sum()):3d} legible "
                  f"({v.label.mean():.1%}), {v.group.nunique()} groups")

    keep = ["file name", "name", "group", "source_family", "label", "split"] + \
           [f"cv_seed{s}" for s in CV_SEEDS]
    df[keep].to_csv(OUT_CSV, index=False)
    print(f"\nSaved {OUT_CSV}")


In [3]:

if __name__ == "__main__":
    main()

1289 signatures, 121 distinct names, 112 groups after merging near-duplicate names
Merged name pairs (check these by eye):
   bakhila ~ gakhila  (0.86)
   bnavya ~ navya  (0.83)
   gayathri ~ pgayathri  (0.89)
   gpreethi ~ spreethi  (0.88)
   kavya ~ navya  (0.8)
   knikhitha ~ nikhitha  (0.89)
   priya ~ sriya  (0.8)
   psuvarna ~ ysuvarna  (0.88)
   shahana ~ sharanya  (0.75)
Group size (rows): min 10, median 11, max 31

Names all illegible: 39, all legible: 4, mixed: 78

Legible rate by source family (possible confound):
               rows  legible_rate
source_family                    
A               400         0.460
B               834         0.221
C                55         0.109

DEV (used for model selection)
  rows 1035, legible 300 (29.0%), groups 89

TEST (touch once, at the very end)
  rows 254, legible 74 (29.1%), groups 23

CV seed 42: fold sizes / legible / groups
   fold 0:  211 rows,  60 legible (28.4%), 19 groups
   fold 1:  200 rows,  61 legible (30.5%), 18 gro